# Step 4 Replay — Generate Real Table Drafts from the Previous Audit ZIP

**Use the output ZIP you already ran**: `tfl_official_library_deepseek_audit.zip`.

This notebook **does not call DeepSeek again** and **does not require a Google Drive-mounted SAP**. It reuses your six previously extracted, evidence-validated Table specifications and SAP evidence records, reruns Step 3 against the official **six-study / 431-template Library**, then calls the newly implemented Step 4 **source-verified draft generator** and Step 5 validation.

**Important interpretation:** Every new DOCX is a clearly labelled **DRAFT**, with empty `xx` data placeholders. The generator verifies a real Library DOCX, its SHA-256 and Word bookmarks, then **reconstructs the target rows, columns, and notes from the target Specification**. It does *not* falsely transplant the original study's clinical contents or claim full bookmark-level adaptation. All drafts require statistical/clinical review.

Your source Library and original audit ZIP are never edited. The ZIP may contain confidential SAP evidence; keep all inputs and outputs private.

## 0 · Clone the development branch, which includes the Step 4 implementation

No manual package ZIP upload. Git clone includes the **official Library** and the updated Agent code. The source templates are only read, not modified.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/wohaofanaweishenme-cell/tfl-generation-agent.git"
GIT_REF = "feature/official-library-colab"
PROJECT_ROOT = Path("/content/tfl-official-step4-replay")
if (PROJECT_ROOT / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "fetch", "origin", GIT_REF])
    subprocess.check_call(["git", "-C", str(PROJECT_ROOT), "checkout", "-B", GIT_REF, "FETCH_HEAD"])
elif not PROJECT_ROOT.exists():
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", GIT_REF,
                           REPO_URL, str(PROJECT_ROOT)])
else:
    raise RuntimeError(f"{PROJECT_ROOT} exists but is not a Git checkout")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)])
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from tfl_agent import TFLSession
from tfl_agent.tools.library_loader import load_library
from tfl_agent.agent.step3_retriever import retrieve_templates, template_selection_decision
from tfl_agent.tools.io_utils import write_json
from IPython.display import display, FileLink
import pandas as pd
import json, zipfile, shutil, traceback

git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("Using Agent + official Library branch:", GIT_REF, "at", git_sha)
print("Python package:", __import__("tfl_agent").__file__)

## 1 · Verify and load the unified official Library

The root folder is `library/`, NOT the old `library/studies/P-VCNA-003/` and NOT `examples/`. The notebook attaches official per-study Bookmark Locator Maps from `Template_Index.json` in memory.

It only exposes approved templates and checks that all registered canonical DOCX paths exist. No metadata or DOCX files are written into the Library.

In [ ]:
LIBRARY_ROOT = PROJECT_ROOT / "library"
registry = json.loads((LIBRARY_ROOT / "Library_Registry.json").read_text(encoding="utf-8"))
entries = json.loads((LIBRARY_ROOT / "Template_Index.json").read_text(encoding="utf-8"))["entries"]
entry_by_id = {x["template_id"]: x for x in entries}
approved = {x["template_id"] for x in entries
            if x.get("review_status") == "approved"
            and x.get("formal_use_eligible") is True}

library = load_library(LIBRARY_ROOT)
for t in library["templates"]:
    tid = t["identity"]["template_id"]
    rec = entry_by_id[tid]
    docx = (PROJECT_ROOT / rec["docx_path"]).resolve()
    locator = (PROJECT_ROOT / rec["locator_map_path"]).resolve()
    assert docx.is_file() and locator.is_file(), f"Missing official source/locator {tid}"
    assert Path(t["resolved_path"]).resolve() == docx, f"Unexpected template path: {tid}"
    t["locator_map_path"] = str(locator)  # In-memory only
library["templates"] = [t for t in library["templates"]
                        if t["identity"]["template_id"] in approved]
library["library_id"] = registry["library_id"]
library["format"] = "official_unified_v2_4_approved"
assert len(library["templates"]) == registry["counts"]["approved"] == 431
assert len(set(t["source"]["study_id"] for t in library["templates"])) == 6
print("Official Library:", library["library_id"])
print("Approved templates:", len(library["templates"]))
display(pd.Series([t["identity"]["output_type"] for t in library["templates"]])
        .value_counts().rename_axis("Type").reset_index(name="Count"))

## 2 · Reuse your previous DeepSeek audit ZIP — no model calls

If you already uploaded the ZIP into this Colab runtime, update `AUDIT_ZIP_PATH`; otherwise run this cell and upload `tfl_official_library_deepseek_audit.zip`. This ZIP holds six real Table specifications (and their original evidence records). It is kept inside this private runtime only.

In [ ]:
from google.colab import files
AUDIT_ZIP_PATH = Path("/content/tfl_official_library_deepseek_audit.zip")
if not AUDIT_ZIP_PATH.is_file():
    uploaded = files.upload()
    if "tfl_official_library_deepseek_audit.zip" in uploaded:
        AUDIT_ZIP_PATH = Path("/content/tfl_official_library_deepseek_audit.zip")
    elif len(uploaded) == 1:
        AUDIT_ZIP_PATH = Path("/content") / next(iter(uploaded))
if not AUDIT_ZIP_PATH.is_file():
    raise FileNotFoundError("Upload the prior audit ZIP or set AUDIT_ZIP_PATH")

AUDIT_DIR = Path("/content/step4_previous_audit")
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(AUDIT_ZIP_PATH) as archive:
    # Prevent untrusted ZIP paths from escaping the audit directory.
    for member in archive.infolist():
        destination = (AUDIT_DIR / member.filename).resolve()
        if not destination.is_relative_to(AUDIT_DIR.resolve()):
            raise ValueError("Unsafe path in supplied audit ZIP: " + member.filename)
    archive.extractall(AUDIT_DIR)
target_dirs = sorted(p for p in AUDIT_DIR.iterdir()
                     if p.is_dir() and (p / "parsed_sap_blocks.json").is_file())
assert target_dirs, "ZIP does not contain per-TFL Step-1/2 audit data."
display(pd.DataFrame([{
    "Target folder": p.name,
    "Has real spec": bool(list((p / "specifications").glob("*.json"))),
    "Has source evidence": (p / "parsed_sap_blocks.json").is_file(),
} for p in target_dirs]))
print("No Google Drive / DeepSeek account required for Step 4 replay.")

## 3 · Replay Step 4 + Step 5 for every saved Table

For every saved target, the notebook loads exactly its prior `specifications/<tfl_id>.json` and `parsed_sap_blocks.json`. It reconstructs the same current Step-3 retrieval pool, selects an endpoint-compatible **layout reference** (never automatically turning a clinical mismatch into an approval), generates a target-spec DOCX draft, and validates it.

For PFS, the draft selector will not choose a source explicitly labelled Overall Survival merely because its numeric ranking is higher. For Hematology, it prefers a Hematology reference over Biochemistry when available. The source reference and retrieval status are recorded in the output.

If the saved Specification was originally `blocked`, its DOCX will explicitly say which target rows, columns or statistics were missing. **Its existence is not a claim of clinical readiness.**

In [ ]:
OUTPUT_ROOT = Path("/content/step4_official_draft_outputs")
records = []
for folder in target_dirs:
    print("\n" + "=" * 75)
    print("SOURCE AUDIT:", folder.name)
    specs = list((folder / "specifications").glob("*.json"))
    if len(specs) != 1:
        print("SKIP: expected exactly one saved target Specification")
        continue
    spec = json.loads(specs[0].read_text(encoding="utf-8"))
    target = spec["identity"]
    tid = target["tfl_id"]
    try:
        session = TFLSession(
            sap_path=Path("/content/unused_cached_SAP.pdf"),  # No SAP read; original parsed blocks reused
            library_path=LIBRARY_ROOT,
            schema_path=PROJECT_ROOT / "schemas" / "TFL_Specification.schema.json",
            prompts_dir=PROJECT_ROOT / "prompts",
            output_dir=OUTPUT_ROOT / tid,
            llm=None,  # No DeepSeek or LLM call
            tfl_id=tid, title=target["title"],
            draft_mode=True,
        )
        session.sap = json.loads((folder / "parsed_sap_blocks.json").read_text(encoding="utf-8"))
        inventory = json.loads((folder / "target_inventory.json").read_text(encoding="utf-8"))
        session.inventory = inventory["targets"]
        session.identification_mode = inventory["identification_mode"]
        session.tfl = next(x for x in session.inventory if x["tfl_id"] == tid)
        session.spec = spec
        session.spec_validation = json.loads(
            (folder / "validation" / f"{tid}_specification.json").read_text(encoding="utf-8")
        )
        session.library = library
        session.matches = retrieve_templates(spec, library, top_k=10)
        session.selection_decision = template_selection_decision(session.matches, min_score=0.55)
        session.selection = session.selection_decision["selection"]
        print("Step 2 / Saved readiness:", spec["readiness"])
        print("Step 3 / Original-style retrieval:", session.selection_decision["status"])
        print("Step 3 / Best-ranked:", session.selection_decision["best_candidate_template_id"])
        preview = session.preview_modification_plan()
        print("Step 4 / Layout reference:", preview["candidate_template_id"])
        result = session.step4_generate_shell()
        print("Step 4 / Status:", result["status"])
        if result.get("issues"):
            print("Step 4 / First review notes:", result["issues"][:3])
        report = session.step5_validate_and_export()
        print("Step 5 / Structural validation:", report["word_validation"])
        print("Step 5 / Clinical approval:", report["clinical_approval"])
        if session.shell_path:
            display(FileLink(session.shell_path, result_html_prefix="Open EDITABLE unapproved DOCX draft: "))
        records.append({
            "Target": tid, "Spec readiness": spec["readiness"],
            "Retrieval status": session.selection_decision["status"],
            "Top-ranked candidate": session.selection_decision["best_candidate_template_id"],
            "Draft layout reference": session.selection["template_id"] if session.selection else None,
            "Step 4": result["status"],
            "DOCX exists": bool(session.shell_path),
            "Structural check": report["word_validation"]["valid"],
        })
    except Exception as exc:
        traceback.print_exc()
        records.append({"Target": tid, "Step 4": "error", "DOCX exists": False,
                        "Error": f"{type(exc).__name__}: {exc}"})

summary = pd.DataFrame(records)
display(summary)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT_ROOT / "step4_replay_summary.csv", index=False)
if not all(r.get("DOCX exists") for r in records):
    print("Some drafts did not generate. Inspect errors above; do not treat this as a complete pass.")
else:
    print("All saved Table specifications produced editable review-required DOCX drafts.")

## 4 · Export the generated DOCX drafts and audit reports

This exports the original cached evidence linkage, each `Table_<id>_DRAFT.docx`, modification plans, validation reports, plus a CSV comparison. **All remain unapproved until target-study review.** No source Library writes and no DeepSeek charges.

In [ ]:
from google.colab import files
ARCHIVE = Path(shutil.make_archive(
    "/content/step4_official_draft_outputs", "zip", root_dir=OUTPUT_ROOT
))
print("Download:", ARCHIVE, "size (MB):", round(ARCHIVE.stat().st_size / 2**20, 2))
files.download(str(ARCHIVE))

## What this Stage B milestone demonstrates

**It proves a runnable Step 3 → Step 4 → Step 5 prototype using real, source-verified DOCX and saved real target SAP specifications.** The output is a target-spec-first reconstruction, **not** a claim that rows/columns/footnotes were mapped one-to-one to the source DOCX's Word bookmarks. The original source DOCX is used for authenticity checking and page setup; statistical content is never silently copied.

A later stage can add approved source-to-target bookmark mapping, advanced table geometry adaptation, denominator logic, independent footnote extraction, and visual/clinical QA. The current Step 2 may still have extraction gaps; the new Step 4 does not mask or fix them.

This replay Notebook intentionally does not run Listing or Figure generation, which is outside the current Table-only Step 4.